FC-CxtA

In [ ]:
# %% Unified FC–Recall (CxtA + CxtB), skip missing, no regression, no scatter

import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import onep  # your 1p analysis utilities

# =========================
# Config
# =========================

# Collections
FC_COLLECTION_PATH    = "/Users/suthardr/Desktop/collection_fc_allmice.pkl"
CXT_A_COLLECTION_PATH = "/Users/suthardr/Desktop/collection_cxta_allmice.pkl"
CXT_B_COLLECTION_PATH = "/Users/suthardr/Desktop/collection_cxtb_allmice.pkl"

# Recall Events (CSV in wide format: one column per animal)
CXT_A_CSV_PATH = (
    "/Volumes/rkc_ramirezlab/Home/rsenne/dCA1_Clean_Data/"
    "revision_analysis/Figure3_Recall/num_events_Fig3/event_times_cxta.csv"
)

CXT_B_CSV_PATH = (
    "/Volumes/rkc_ramirezlab/Home/rsenne/dCA1_Clean_Data/"
    "revision_analysis/Figure3_Recall/num_events_Fig3/event_times_cxtb.csv"
)

# Output dirs
BASE_FIG_DIR   = "/Users/suthardr/Desktop/Fig4"
HEATMAP_DIR_A  = os.path.join(BASE_FIG_DIR, "RecallA_sorted_FC_argmax")
HEATMAP_DIR_B  = os.path.join(BASE_FIG_DIR, "RecallB_sorted_FC_argmax")

os.makedirs(HEATMAP_DIR_A, exist_ok=True)
os.makedirs(HEATMAP_DIR_B, exist_ok=True)

# FC events & trace length
FC_EVENTS  = [120, 180, 240, 300]
N_SAMPLES  = 3303  # truncate traces & timestamps


# =========================
# Helpers
# =========================

def load_collections():
    """Load FC, CxtA, and CxtB collections."""
    with open(FC_COLLECTION_PATH, "rb") as f:
        collection_fc = pickle.load(f)
    with open(CXT_A_COLLECTION_PATH, "rb") as f:
        collection_cxta = pickle.load(f)
    with open(CXT_B_COLLECTION_PATH, "rb") as f:
        collection_cxtb = pickle.load(f)
    return collection_fc, collection_cxta, collection_cxtb


def load_recall_events(csv_path, suffix):
    """
    Return dict animal_id -> list of recall event times.
    Assumes wide CSV, each column = animal+suffix (e.g. 'astroF3_cxta').
    """
    df = pd.read_csv(csv_path)
    events_by_animal = {}
    for col in df.columns:
        animal = col.replace(suffix, "")
        events = df[col].dropna().tolist()
        events_by_animal[animal] = events
    return events_by_animal


def plot_heatmaps(sorted_arr_fc, sorted_arr_recall,
                  time_fc, time_recall,
                  animal, context_label, out_dir):
    """Save side-by-side FC and Recall heatmaps."""
    first_last_fc = [0, sorted_arr_fc.shape[0]]
    first_last_recall = [0, sorted_arr_recall.shape[0]]

    fig, axs = plt.subplots(1, 2, figsize=(10, 7), sharey=True)

    # FC heatmap
    sns.heatmap(
        sorted_arr_fc, cmap="mako", cbar=False, ax=axs[0],
        vmin=-3, vmax=6, rasterized=True, annot=False
    )
    axs[0].axvline((time_fc.shape[0] / 3), linestyle="--", color="white")
    axs[0].tick_params(axis="both", which="major", labelsize=12)
    axs[0].set_xticks(np.linspace(0, sorted_arr_fc.shape[1], 5))
    axs[0].set_xticklabels(
        np.linspace(time_fc.min(), time_fc.max(), 5).astype(int),
        rotation=0
    )
    axs[0].set_yticks(first_last_fc)
    axs[0].set_yticklabels(first_last_fc, rotation=0)
    axs[0].set_title("FC Sorted by FC Argmax", fontsize=14)
    axs[0].set_xlabel("Time (s)", fontsize=14)
    axs[0].set_ylabel("Reactivated Cell #", fontsize=14)

    # Recall heatmap
    heatmap = sns.heatmap(
        sorted_arr_recall, cmap="mako", cbar=True, rasterized=True,
        cbar_kws={"label": r"z-scored $\frac{dF}{F}$"},
        ax=axs[1], vmin=-3, vmax=6, annot=False
    )
    axs[1].axvline((time_recall.shape[0] / 3), linestyle="--", color="white")
    axs[1].tick_params(axis="both", which="major", labelsize=12)
    axs[1].set_xticks(np.linspace(0, sorted_arr_recall.shape[1], 5))
    axs[1].set_xticklabels(
        np.linspace(time_recall.min(), time_recall.max(), 5).astype(int),
        rotation=0
    )
    axs[1].set_yticks(first_last_recall)
    axs[1].set_yticklabels(first_last_recall, rotation=0)
    axs[1].set_title(f"Recall {context_label} Sorted by FC Argmax", fontsize=14)
    axs[1].set_xlabel("Time (s)", fontsize=14)

    cbar = heatmap.collections[0].colorbar
    cbar.ax.yaxis.set_tick_params(labelsize=12)
    cbar.set_label(r"z-scored $\frac{dF}{F}$", fontsize=14)

    plt.tight_layout()
    fname = f"{animal}_{context_label}_reactivated_sort_sequence.svg"
    fig.savefig(os.path.join(out_dir, fname))
    plt.close(fig)


def process_animal(
    animal,
    collection_fc,
    collection_recall,   # cxta or cxtb
    recall_events,       # dict animal -> list of event times
    reg_key,             # "FC-A" or "FC-B"
    heatmap_dir,
    context_label        # "CxtA" or "CxtB"
):
    """
    Run the full FC–Recall pipeline for one animal + one context.
    Returns dict with summary stats or None if skipped.
    """
    print(f"\nProcessing {animal} ({context_label})...")

    # --- recall events (from correct CSV for this context) ---
    events_recall = recall_events.get(animal)
    if not events_recall:
        print("  No recall events in CSV, skipping.")
        return None

    # --- registration table (FC vs recall) ---
    try:
        reg = collection_fc.animals[animal].registration_tables[reg_key]
    except Exception as e:
        print(f"  No registration table {reg_key} for {animal}: {e}")
        return None

    reg = pd.DataFrame(reg)

    # --- ensure recall traces for this animal exist in this collection ---
    if animal not in collection_recall.animals:
        print(f"  Recall traces for {animal} not found in this collection ({context_label}), skipping.")
        return None

    # --- rejected indices ---
    rejected_fc = pd.Series(collection_fc.animals[animal].rejected_inds)
    rejected_recall = pd.Series(collection_recall.animals[animal].rejected_inds)

    reg_fc_ok = reg[~reg[0].isin(rejected_fc)]
    reg_both_ok = reg_fc_ok[~reg_fc_ok[1].isin(rejected_recall)]

    # keep only valid (non -1) indices
    both_idxs = reg_both_ok.loc[
        (reg_both_ok[0] != -1) & (reg_both_ok[1] != -1)
    ]
    if both_idxs.shape[0] == 0:
        print("  No overlapping good cells after rejection/mapping, skipping.")
        return None

    # --- traces ---
    fc_traces = collection_fc.animals[animal].get_traces()
    recall_traces = collection_recall.animals[animal].get_traces()

    # robustly ensure registration indices are within bounds of traces
    n_fc = len(fc_traces)
    n_recall = len(recall_traces)

    valid_mask = (both_idxs[0] < n_fc) & (both_idxs[1] < n_recall)
    both_valid = both_idxs.loc[valid_mask]

    if both_valid.shape[0] == 0:
        print(
            f"  All overlapping cells are out of bounds "
            f"(n_fc={n_fc}, n_recall={n_recall}), skipping."
        )
        return None

    fc_idx = both_valid[0].astype(int).to_numpy()
    recall_idx = both_valid[1].astype(int).to_numpy()

    fc_traces_arr = np.array(fc_traces)
    recall_traces_arr = np.array(recall_traces)

    fc_active = fc_traces_arr[fc_idx, :N_SAMPLES]
    recall_active = recall_traces_arr[recall_idx, :N_SAMPLES]

    # --- z-score across time ---
    fc_dfz = stats.zscore(fc_active, axis=1)
    recall_dfz = stats.zscore(recall_active, axis=1)

    # --- timestamps ---
    fc_ts = collection_fc.animals[animal].Timestamps[:N_SAMPLES].to_numpy().squeeze()
    recall_ts = collection_recall.animals[animal].Timestamps[:N_SAMPLES].to_numpy().squeeze()

    # --- event-triggered averages ---
    across_fc, time_fc = onep.eta_individual_cells(
        data=fc_dfz,
        timestamps=fc_ts,
        events=[FC_EVENTS],
        window=28,
    )
    across_recall, time_recall = onep.eta_individual_cells(
        data=recall_dfz,
        timestamps=recall_ts,
        events=[events_recall],
        window=28,
    )

    # --- sort by FC argmax ---
    max_idxs = np.argmax(across_fc, axis=1)
    sort_order = np.argsort(max_idxs)

    sorted_fc = across_fc[sort_order]
    sorted_recall = across_recall[sort_order]

    # --- plots ---
    plot_heatmaps(sorted_fc, sorted_recall, time_fc, time_recall,
                  animal, context_label, heatmap_dir)

    # --- argmax times per cell ---
    argmax_fc = time_fc[np.argmax(sorted_fc, axis=1)]
    argmax_recall = time_recall[np.argmax(sorted_recall, axis=1)]

    # --- stats ---
    rho, p_spear = stats.spearmanr(argmax_fc, argmax_recall)
    print(f"  Spearman rho = {rho:.3f}, p = {p_spear:.3g}")
    print(f"  Finished {animal} ({context_label}) with {len(argmax_fc)} overlapping cells.")

    # store argmaxes as strings so they survive CSV save (optional)
    return dict(
        animal=animal,
        context=context_label,
        n_cells=len(argmax_fc),
        spearman_rho=rho,
        spearman_p=p_spear,
        argmax_fc=";".join(map(str, argmax_fc)),
        argmax_recall=";".join(map(str, argmax_recall)),
    )


# =========================
# Main
# =========================

collection_fc, collection_cxta, collection_cxtb = load_collections()

recall_events_cxta = load_recall_events(CXT_A_CSV_PATH, "_cxta")
recall_events_cxtb = load_recall_events(CXT_B_CSV_PATH, "_cxtb")

print("Recall events found for CxtA animals:")
for k, v in recall_events_cxta.items():
    print(f"  {k}: {v}")

print("\nRecall events found for CxtB animals:")
for k, v in recall_events_cxtb.items():
    print(f"  {k}: {v}")

# Build unified mouse list: mice that have FC and at least one recall context
mice_fc   = set(collection_fc.animals.keys())
mice_cxta = set(recall_events_cxta.keys())
mice_cxtb = set(recall_events_cxtb.keys())

ALL_MICE = sorted(mice_fc & (mice_cxta | mice_cxtb))
print("\nWill attempt these mice:", ALL_MICE)

results = []

for animal in ALL_MICE:

    # ---------- FC–CxtA ----------
    if (animal in recall_events_cxta and
        "FC-A" in collection_fc.animals[animal].registration_tables):

        summaryA = process_animal(
            animal=animal,
            collection_fc=collection_fc,
            collection_recall=collection_cxta,
            recall_events=recall_events_cxta,   # events from CxtA CSV
            reg_key="FC-A",
            heatmap_dir=HEATMAP_DIR_A,
            context_label="CxtA",
        )
        if summaryA is not None:
            results.append(summaryA)

    # ---------- FC–CxtB ----------
    if (animal in recall_events_cxtb and
        "FC-B" in collection_fc.animals[animal].registration_tables):

        summaryB = process_animal(
            animal=animal,
            collection_fc=collection_fc,
            collection_recall=collection_cxtb,
            recall_events=recall_events_cxtb,   # events from CxtB CSV
            reg_key="FC-B",
            heatmap_dir=HEATMAP_DIR_B,
            context_label="CxtB",
        )
        if summaryB is not None:
            results.append(summaryB)

df_results = pd.DataFrame(results)
print("\nSummary across animals and contexts:")
print(df_results)

out_path = os.path.join(BASE_FIG_DIR, "FC_crossval_reactivation_summary_CxtA_CxtB.csv")
df_results.to_csv(out_path, index=False)
print(f"\nSaved df_results to: {out_path}")


In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

def shuffle_control_test(x, y, n_shuffles=1000, seed=None, two_sided=True):
    x = np.asarray(x); y = np.asarray(y)
    if x.shape[0] != y.shape[0]:
        raise ValueError(f"Length mismatch: x={x.shape[0]}, y={y.shape[0]}")
    if not np.all(np.isfinite(x)) or not np.all(np.isfinite(y)):
        raise ValueError("Non-finite values in x or y")
    rng = np.random.default_rng(seed)
    r_obs, p_obs = stats.spearmanr(x, y)
    sh = np.empty(n_shuffles, dtype=float)
    for i in range(n_shuffles):
        y_perm = rng.permutation(y)
        sh[i], _ = stats.spearmanr(x, y_perm)
    if two_sided:
        p_emp = (np.sum(np.abs(sh) >= abs(r_obs)) + 1) / (n_shuffles + 1)
    else:
        if r_obs >= 0:
            p_emp = (np.sum(sh >= r_obs) + 1) / (n_shuffles + 1)
        else:
            p_emp = (np.sum(sh <= r_obs) + 1) / (n_shuffles + 1)
    return {
        "r_obs": float(r_obs),
        "p_obs_parametric": float(p_obs),
        "shuffle_rhos": sh,
        "p_empirical": float(p_emp),
        "null_mean": float(np.mean(sh)),
        "null_std": float(np.std(sh, ddof=1)),
        "z_obs_vs_null": float((r_obs - np.mean(sh)) / (np.std(sh, ddof=1) + 1e-12)),
    }

def plot_shuffle_null(shuffle_rhos, r_obs, figsize=(3.5, 3.5), dpi=600):
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    ax.hist(shuffle_rhos, bins=30, density=True)
    ax.axvline(r_obs, linestyle='--')
    ax.set_xlabel('Spearman ρ')
    ax.set_ylabel('Density')
    ax.set_title('Shuffle null (permute Recall across cells)')
    return fig, ax


In [ ]:
import pandas as pd
import os

def run_fig4_fc_recall_shuffle(
    fig4_csv_path,
    save_dir,
    n_shuffles=10000,
    two_sided=True
):
    os.makedirs(save_dir, exist_ok=True)
    df = pd.read_csv(fig4_csv_path)

    results = []

    for idx, row in df.iterrows():
        animal  = row["animal"]
        context = row["context"] if "context" in row else "NA"

        # parse argmax lists: "12.3;15.8;..." -> np.array
        try:
            fc_times     = np.array([float(x) for x in str(row["argmax_fc"]).split(";") if x != ""])
            recall_times = np.array([float(x) for x in str(row["argmax_recall"]).split(";") if x != ""])
        except Exception as e:
            print(f"[WARN] {animal} {context}: failed to parse argmax lists: {e}. Skipping.")
            continue

        if fc_times.size != recall_times.size or fc_times.size < 2:
            print(f"[WARN] {animal} {context}: size mismatch or <2 cells (n={fc_times.size}). Skipping.")
            continue

        # observed correlation
        r_obs, p_obs = stats.spearmanr(fc_times, recall_times)
        print(f"{animal} {context}: observed rho={r_obs:.3f}, p={p_obs:.3g}, n_cells={fc_times.size}")

        # shuffle null
        seed = hash(f"{animal}_{context}") & 0xFFFFFFFF
        sh = shuffle_control_test(
            fc_times,
            recall_times,
            n_shuffles=n_shuffles,
            seed=seed,
            two_sided=two_sided
        )

        # optional: histogram plot per animal/context
        fig, ax = plot_shuffle_null(sh["shuffle_rhos"], sh["r_obs"])
        hist_path = os.path.join(
            save_dir,
            f"Fig4_FCvsRecall_shuffle_null_{animal}_{context}.svg"
        )
        fig.savefig(hist_path, dpi=600, bbox_inches="tight")
        plt.close(fig)

        results.append({
            "animal": animal,
            "context": context,
            "n_cells": fc_times.size,
            "rho_obs": sh["r_obs"],
            "p_obs_parametric": sh["p_obs_parametric"],
            "p_empirical": sh["p_empirical"],
            "null_mean_rho": sh["null_mean"],
            "null_std_rho": sh["null_std"],
            "z_obs_vs_null": sh["z_obs_vs_null"],
            "hist_path": hist_path,
        })

    res_df = pd.DataFrame(results)
    out_csv = os.path.join(save_dir, "Fig4_FCvsRecall_shuffle_summary.csv")
    res_df.to_csv(out_csv, index=False)
    print(f"\nSaved Fig4 shuffle summary to: {out_csv}")
    return res_df


In [ ]:
fig4_csv_path = "/Users/suthardr/Desktop/Fig4/FC_crossval_reactivation_summary_CxtA_CxtB.csv"
save_dir      = "/Users/suthardr/Desktop/Fig4/shuffle_FC_vs_recall"

res_fig4 = run_fig4_fc_recall_shuffle(
    fig4_csv_path=fig4_csv_path,
    save_dir=save_dir,
    n_shuffles=10000,
    two_sided=True
)

res_fig4


In [ ]:
def boxpaired_two_contexts_vs_shuffle(
    df_a, df_b,
    id_col="animal",
    obs_col="spearman_rho",
    null_col="shuffle_null_mean",
    title="Contexts: ρ vs shuffle",
    ylim=(0, 1),
    palette=("#C9D6DF", "#B39BC8", "#C9D6DF", "#C79BC8"),  # A: shuffle, A: obs, B: shuffle, B: obs
    dpi=600,
    save_path=None,
):
    """
    Single-axis plot with four boxes:
      A - Shuffle, A - Observed, B - Shuffle, B - Observed
    Draws paired lines within A and within B (no stats, no text).
    """
    # Prepare long-form data
    A = df_a[[id_col, obs_col, null_col]].dropna().copy()
    B = df_b[[id_col, obs_col, null_col]].dropna().copy()

    A_long = A.melt(id_vars=id_col, value_vars=[null_col, obs_col],
                    var_name="condition", value_name="rho")
    B_long = B.melt(id_vars=id_col, value_vars=[null_col, obs_col],
                    var_name="condition", value_name="rho")

    name_map = {obs_col: "Observed", null_col: "Shuffle"}
    A_long["condition"] = A_long["condition"].map(name_map)
    B_long["condition"] = B_long["condition"].map(name_map)
    A_long["label"] = A_long["condition"].map({"Shuffle": "A - Shuffle", "Observed": "A - Observed"})
    B_long["label"] = B_long["condition"].map({"Shuffle": "B - Shuffle", "Observed": "B - Observed"})

    long = pd.concat([A_long, B_long], ignore_index=True)

    order = ["A - Shuffle", "A - Observed", "B - Shuffle", "B - Observed"]
    pal = {lab: col for lab, col in zip(order, palette)}
    x_pos = {lab: i for i, lab in enumerate(order)}  # category positions

    # Style + figure
    plt.rcParams.update({'font.size': 8, 'font.family': 'Arial'})
    fig, ax = plt.subplots(figsize=(5.0, 3.5), dpi=dpi)

    # Boxes only
    sns.boxplot(
        data=long, x="label", y="rho",
        order=order, palette=pal, ax=ax,
        width=0.6, showfliers=False
    )

    # Paired lines for A
    A_pairs = A_long.pivot_table(index=id_col, columns="condition", values="rho")
    if {"Shuffle", "Observed"}.issubset(A_pairs.columns):
        for _, row in A_pairs.iterrows():
            ax.plot([x_pos["A - Shuffle"], x_pos["A - Observed"]],
                    [row["Shuffle"], row["Observed"]],
                    color="gray", alpha=0.7, linewidth=1)
            ax.scatter([x_pos["A - Shuffle"], x_pos["A - Observed"]],
                       [row["Shuffle"], row["Observed"]],
                       s=10, color="black", zorder=3)

    # Paired lines for B
    B_pairs = B_long.pivot_table(index=id_col, columns="condition", values="rho")
    if {"Shuffle", "Observed"}.issubset(B_pairs.columns):
        for _, row in B_pairs.iterrows():
            ax.plot([x_pos["B - Shuffle"], x_pos["B - Observed"]],
                    [row["Shuffle"], row["Observed"]],
                    color="gray", alpha=0.7, linewidth=1)
            ax.scatter([x_pos["B - Shuffle"], x_pos["B - Observed"]],
                       [row["Shuffle"], row["Observed"]],
                       s=10, color="black", zorder=3)

    # Cosmetics
    ax.set_xlabel("")
    ax.set_ylabel("Spearman's ρ", labelpad=8)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.set_title(title, pad=8)
    sns.despine(ax=ax)
    plt.tight_layout(pad=3)

    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight", format=save_path.split(".")[-1])

    return fig, ax

In [ ]:
fig4_fp = "/Users/suthardr/Desktop/Fig4/shuffle_FC_vs_recall/Fig4_FCvsRecall_shuffle_summary.csv"
df4 = pd.read_csv(fig4_fp)

# CxtA animals with >10 cells
cxta_fig4 = df4[
    (df4["context"] == "CxtA") &
    (df4["n_cells"] > 9)
].copy()

# CxtB animals with >10 cells
cxtb_fig4 = df4[
    (df4["context"] == "CxtB") &
    (df4["n_cells"] >9 )
].copy()

print("CxtA animals kept:", cxta_fig4["animal"].unique())
print("CxtB animals kept:", cxtb_fig4["animal"].unique())


In [ ]:
fig, ax = boxpaired_two_contexts_vs_shuffle(
    cxta_fig4,
    cxtb_fig4,
    id_col="animal",
    obs_col="rho_obs",          # or "spearman_rho" depending on your column name
    null_col="null_mean_rho",
    title="Context A/B: Observed vs Shuffle (n_cells > 10)",
    ylim=(-0.5, 0.5),
    save_path="/Users/suthardr/Desktop/Fig4/ctxAB_rho_vs_shuffle_gt10cells.svg"
)


In [ ]:
import pandas as pd

fig4_fp = "/Users/suthardr/Desktop/Fig4/shuffle_FC_vs_recall/Fig4_FCvsRecall_shuffle_summary.csv"
df4 = pd.read_csv(fig4_fp)

# Split by context
cxta = df4[df4["context"] == "CxtA"].copy()
cxtb = df4[df4["context"] == "CxtB"].copy()

# Rename columns to match Fig3 convention
rename_map = {
    "rho_obs": "spearman_rho",
    "null_mean_rho": "shuffle_null_mean",
    "null_std_rho": "shuffle_null_std",
    "z_obs_vs_null": "shuffle_z",
}

cxta_compat = cxta.rename(columns=rename_map)
cxtb_compat = cxtb.rename(columns=rename_map)

# (optional) reorder columns like Fig3
cols_order = ["animal", "spearman_rho", "shuffle_null_mean", "shuffle_null_std", "shuffle_z"]
cols_order = [c for c in cols_order if c in cxta_compat.columns]  # safety

cxta_compat = cxta_compat[cols_order + [c for c in cxta_compat.columns if c not in cols_order]]
cxtb_compat = cxtb_compat[cols_order + [c for c in cxtb_compat.columns if c not in cols_order]]

# Save in the exact locations/names you used for Fig3
cxta_out_fp = "/Users/suthardr/Desktop/Revision/crossval_summary_results_cxta.csv"
cxtb_out_fp = "/Users/suthardr/Desktop/Revision/crossval_summary_results_cxtb.csv"

cxta_compat.to_csv(cxta_out_fp, index=False)
cxtb_compat.to_csv(cxtb_out_fp, index=False)

print("Saved:")
print("  ", cxta_out_fp)
print("  ", cxtb_out_fp)


In [ ]:
def compare_rho_vals(cxta_fp, cxtb_fp):
    # read in both csvs
    cxta = pd.read_csv(cxta_fp)
    cxtb = pd.read_csv(cxtb_fp)

    # extract columns we need
    cxta = cxta[["animal", "spearman_rho", "shuffle_null_mean", "shuffle_null_std", "shuffle_z"]]
    cxtb = cxtb[["animal", "spearman_rho", "shuffle_null_mean", "shuffle_null_std", "shuffle_z"]]

    # paired t-test observed vs null within each context
    stata, pa = stats.ttest_rel(cxta["spearman_rho"], cxta["shuffle_null_mean"])
    statb, pb = stats.ttest_rel(cxtb["spearman_rho"], cxtb["shuffle_null_mean"])

    # between-context comparison of (obs - null)
    diff_a = cxta["spearman_rho"] - cxta["shuffle_null_mean"]
    diff_b = cxtb["spearman_rho"] - cxtb["shuffle_null_mean"]
    stat_diff, p_diff = stats.ttest_ind(diff_a, diff_b, equal_var=False)

    return pa, pb, p_diff, stata, statb, stat_diff


In [ ]:
from scipy import stats

pa, pb, p_diff, stata, statb, stat_diff = compare_rho_vals(
    "/Users/suthardr/Desktop/Revision/crossval_summary_results_cxta.csv",
    "/Users/suthardr/Desktop/Revision/crossval_summary_results_cxtb.csv"
)

print("CxtA observed vs shuffle p =", pa, "  t =", stata)
print("CxtB observed vs shuffle p =", pb, "  t =", statb)
print("Between-contexts (A vs B in Δρ) p =", p_diff, "  t =", stat_diff)
